In [10]:
import numpy as np
import plotly.graph_objects as go

# --- Core function ---
def flow_rate_cfm(power_kw, rho, cp, delta_T):
    """
    power_kw : array or float (kW)
    rho      : density (kg/m^3)
    cp       : specific heat (J/kg-K)
    delta_T  : temperature rise (K)
    
    returns volumetric flow rate in CFM
    """
    power_w = np.array(power_kw) * 1000  # convert kW → W
    
    # volumetric flow (m^3/s)
    flow_m3_s = power_w / (rho * cp * delta_T)
    
    # convert m^3/s → CFM
    flow_cfm = flow_m3_s * 2118.88
    
    return flow_cfm


# --- Coolant properties (constant ΔT = 20°C) ---
coolants = {
    "Air": {
        "rho": 1.2,
        "cp": 1005,
        "delta_T": 20
    },
    "Water (Direct-to-Chip)": {
        "rho": 1000,
        "cp": 4186,
        "delta_T": 20
    },
    "Oil (Immersion)": {
        "rho": 800,
        "cp": 2000,
        "delta_T": 20
    },
    "PFAS (Immersion)": {
        "rho": 1600,
        "cp": 1200,
        "delta_T": 20
    }
}


# --- Power sweep (0 to 100 kW) ---
power_range = np.linspace(0, 100, 100)


# --- Compute results ---
results = {}

for name, props in coolants.items():
    results[name] = flow_rate_cfm(
        power_range,
        props["rho"],
        props["cp"],
        props["delta_T"]
    )


# --- Plot with Plotly ---
fig = go.Figure()

for name, flow in results.items():
    fig.add_trace(go.Scatter(
        x=power_range,
        y=flow,
        mode='lines',
        name=name
    ))

fig.update_layout(
    title="Flow Rate (CFM) vs Rack Power Density",
    xaxis_title="Rack Power Density (kW)",
    yaxis_title="Flow Rate (CFM) in logscale",
    template="plotly_white"
)
fig.update_yaxes(type="log")
fig.show()